# Ingest - many tables in ONE job task

One `lakefed_ingest` run (a *sweep*) processes a whole **task_collection** of (tenant db, table)
tasks in this single notebook. A thread pool and Spark **FAIR scheduler pools** overlap each
table's work on the cluster. This decouples throughput from the workspace job limits
(2,000 concurrent task runs; 10,000 run-submits/hour) that a task-per-table design hits. The
throttle is compute (cluster cores), which is the intended cost model. Shard by
`task_collection` for horizontal scale: run one sweep per collection.

Notes:
- **One concurrency knob, `parallelism`** = the most units of query work running at once in this
  run, of ANY type: a cursor read+MERGE, a non-partitioned seed, ONE partition of a partitioned seed, a
  small source query (high watermark / size / bounds), or the staging->target swap. Each unit
  holds one slot of a shared `parallel.WorkSlots(parallelism)`. Threads may outnumber slots, but a
  thread only runs Spark/JDBC work while holding a slot, and never holds one while waiting (e.g. a
  table waiting for its partitions), so nested fan-out can't deadlock. Each slot is its own FAIR pool
  (`pool0..pool{parallelism-1}`). The budget is per run: concurrent sweeps of different collections
  on separate job clusters each have their own.
- Cluster must run the **FAIR** scheduler (`spark.scheduler.mode=FAIR`) for the pools to apply.
- **Strategies and sources:** each control row's `load_type` selects a load strategy
  (`load_strategies.py`: `cursor` and `full`) and its `src_type` a source adapter
  (`source_adapters.py`: SQL Server, Oracle, PostgreSQL, MySQL). A row whose pair is not
  supported fails on its own (stage `plan`) with a clear message. `full` rows reload every cycle.
- Source reads live in **source_io.py** (Spark JDBC, or `remote_query` through a UC connection; no
  SQL warehouse). Sink writes live in **sink_writer.py**. This notebook is orchestration, Lakebase
  checkpointing and telemetry.
- One source read per table (per range for partitioned seeds), no `.cache()`: the seed streams the
  JDBC read into the Delta write and the cursor increment feeds the window read straight into the
  MERGE. Partitioned seeds load a staging table and swap it into the target in one atomic commit, so
  readers never see a partial table.
- Because the per-table work runs in-process (no Jobs-UI task per table), progress/telemetry is
  written to Lakebase: one `ingest_run` row per run and one `ingest_event` row per (run, table),
  queryable live and read directly from the Lakebase UC catalog (see docs/monitoring.md).
- Per-table failures are isolated (one bad table does not fail the collection) and recorded to
  both `ingest_event` (with the stage it failed at) and the `checkpoint` status for triage.
- **Table retries** (`table_retries` > 0, off by default): a table that fails with a transient source
  error (a refused or dropped connection, a timeout, throttling) is tried again in the same run after
  a pause that doubles each time, within a per-run budget (`table_retry_budget`) and a stop per
  source server (table_retry.py). Each attempt is its own `ingest_event` row (`retried`, then `ok` or
  `failed`, with `retry_count`).

In [ ]:
# Cluster task. Bootstrap: put this notebook's dir on sys.path and import the shared modules.
import sys, os, json

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)

import state_store as ss
from state_store import StateStore
import cursor_engine
import load_strategies
import parallel
import planner
import sink_writer
import source_io
import table_retry
import worker

In [ ]:
# Params + connect to Lakebase (SDK-minted OAuth token; no stored password), then open a run.
def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


task_collection = widget("task_collection", "example_cohort")
db_instance = widget("db_instance", "lakefed-ingest")
db_name = widget("db_name", "lakefed_ingest")
# Max concurrent units of query work in this run (all types share it; see the header).
parallelism = int(widget("parallelism", "32"))
# Kept as a job parameter; nothing reads it since change detection left with Change Tracking.
detection_chunk = int(widget("detection_chunk", "200"))
# Source credentials: per-DB JSON secrets in this scope (Databricks- or Azure Key Vault-backed);
# each control row names its own secret_key. TLS trust is configurable (prod: "false").
secret_scope = widget("secret_scope", "lakefed_ingest")
trust_server_certificate = widget("trust_server_certificate", "true")
# Onboarding: at most this many seeds per run (the rest wait for later runs). 0 = no limit.
max_seeds_per_run = int(widget("max_seeds_per_run", "0"))
# Kept as a job parameter; the quiet-table bulk advance that reads it returns with the cursor
# engine. 0 = stamp every unchanged table every run.
freshness_interval_s = float(widget("freshness_interval_s", "0"))
# Concurrent queries per source server, across every run sharing this Lakebase database: each run
# takes its share, this divided by the runs reading the server now, at least 1. 0 = no cap.
max_queries_per_source = int(widget("max_queries_per_source", "0"))
# Temporal cursors: lo is pulled back by this many seconds (a row's cursor_lookback_s overrides it),
# and hi is capped at the source clock plus cursor_skew_s (cursor_engine.py).
cursor_lookback_s = int(widget("cursor_lookback_s", "3600"))
cursor_skew_s = int(widget("cursor_skew_s", "300"))
# Table retries: extra attempts per table after a transient source error ("0" = off), and retries
# in the whole run before failures fail at once (table_retry.py).
table_retries = int(widget("table_retries", "0"))
table_retry_budget = int(widget("table_retry_budget", str(table_retry.DEFAULT_BUDGET)))
os.environ["LAKEFED_SQL_SECRET_SCOPE"] = secret_scope
os.environ["LAKEFED_SQL_TRUST_SERVER_CERT"] = trust_server_certificate
# Single source read per table: Delta MERGE materializes a non-Delta (JDBC) source itself when
# this is "auto" (the default) - we add no .cache() of our own. Logged once for visibility.
# (spark.conf.get validates a supplied default against the allowed values, so read it bare.)
try:
    _materialize = spark.conf.get("spark.databricks.delta.merge.materializeSource")
except Exception:
    _materialize = "auto (unset; runtime default)"
print("spark.databricks.delta.merge.materializeSource =", _materialize)

# reconnect: a dropped state connection is reopened (fresh token) instead of failing the run.
store = StateStore.connect_lakebase(db_instance, db_name, reconnect=True)

# Schema upgrade: a database behind this code's schema version (deployed without running
# lakefed_ingest_setup) is migrated here, once, in one transaction; concurrent runs of other
# collections wait on the schema lock and find it done. So upgrading is a deploy.
_schema_have = store.schema_version()
if _schema_have < ss.SCHEMA_VERSION:
    with open("/Workspace" + nb_dir + "/lakebase_schema.sql") as f:
        _ddl = f.read()
    try:
        _migrated = store.migrate(_ddl)
    except Exception as e:
        raise RuntimeError(
            f"the Lakebase schema is at version {_schema_have} and this code needs {ss.SCHEMA_VERSION}; "
            f"applying the upgrade failed and changed nothing: {str(e)[:500]}. Fix the cause, or run the "
            "lakefed_ingest_setup job as the schema's owner, and re-run.") from e
    if _migrated:
        print(f"migrated the Lakebase schema from version {_migrated['from']} to {ss.SCHEMA_VERSION}"
              + (f" ({len(_migrated['renames'])} renames)" if _migrated["renames"] else ""))
elif _schema_have > ss.SCHEMA_VERSION:
    print(f"[schema] the Lakebase schema is at version {_schema_have}, newer than this code's "
          f"{ss.SCHEMA_VERSION}: deploy the newer code")


def _safe(fn, default=""):
    try:
        return fn() or default
    except Exception:
        return default


job_run_id = _safe(lambda: str(ctx.jobRunId().get()))
cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "")
# Per-collection run lock: different task_collections sweep concurrently; a second run of the SAME
# collection (e.g. a schedule firing while the previous sweep is still going) exits as 'skipped'.
# Session-level advisory lock on a DEDICATED connection held for the whole sweep — released
# automatically if the connection drops, so a crashed run never leaves a stale lock.
lock_store = StateStore.connect_lakebase(
    db_instance, db_name, application_name=ss.lock_application_name(task_collection, job_run_id))
if not lock_store.try_lock_collection(task_collection):
    lock_store.close()
    skipped_id = store.start_run(task_collection, job_run_id, cluster_id, parallelism)
    store.finish_run(skipped_id, total=0, ok=0, failed=0, status="skipped",
                       detail="another sweep of this task_collection is already running")
    print(f"run {skipped_id}: SKIPPED - task_collection {task_collection} is already being swept")
    dbutils.notebook.exit(json.dumps({"run_id": skipped_id, "task_collection": task_collection,
                                      "status": "skipped"}))

configs = store.list_task_configs(task_collection)

# Lost-lock guard: if the lock connection drops mid-sweep (network blip, Lakebase restart), the lock
# is released and another run could start. Workers re-check it at most every 30s and stop STARTING
# new tables once it is gone; skipped tables are simply picked up by the next sweep.
import threading, time
_lock_state = {"checked": time.monotonic(), "held": True}
_lock_mu = threading.Lock()


def assert_lock_held(every_s=30):
    with _lock_mu:
        if _lock_state["held"] and time.monotonic() - _lock_state["checked"] >= every_s:
            _lock_state["checked"] = time.monotonic()
            _lock_state["held"] = lock_store.is_alive()
        if not _lock_state["held"]:
            raise RuntimeError("collection lock lost; table deferred to the next sweep")
run_id = store.start_run(task_collection, job_run_id, cluster_id, parallelism)
print(f"run {run_id}: collection {task_collection} -> {len(configs)} tables (parallelism={parallelism})")
# This run holds the collection's lock, so the collection's other open runs are dead (cancelled, or
# their cluster went away): close them and their in-flight table events, which would otherwise stay
# 'running' in the monitoring views forever. Telemetry only: a failure here never stops the sweep.
try:
    closed_runs, closed_events = store.close_abandoned_runs(task_collection, run_id)
    if closed_runs:
        print(f"closed {closed_runs} earlier run(s) of {task_collection} that ended without finishing "
              f"({closed_events} table events left running)")
except Exception as e:
    print(f"[telemetry] could not close earlier unfinished runs: {str(e)[:200]}")

In [ ]:
# Per-(tenant db, table) work runs in worker.ingest_one (src/lakefed_ingest/worker.py, unit-tested:
# the checkpoint advances only after the sink write commits). This cell builds its dependencies:
# the retry policy, the shared slot budget + FAIR pools, the per-source caps, and the unit executor.
from functools import partial
from types import SimpleNamespace
from concurrent.futures import ThreadPoolExecutor

retry_policy = table_retry.RetryPolicy(retries=table_retries, budget=table_retry_budget)


def set_pool(pool):
    # Called when a unit acquires a slot: that unit's Spark jobs run in the slot's FAIR pool.
    spark.sparkContext.setLocalProperty("spark.scheduler.pool", pool)


# ONE budget for all query work in this run, plus a shared executor that runs partition units.
# Table workers (run_parallel below) only orchestrate: they hold a slot just while doing a unit
# themselves and never while waiting on their partitions, so slots can't deadlock.
# max_queries_per_source: this run's share of each server's cap, recounted every 15 s from the runs
# reading the server (shared advisory locks held on the lock connection, released with it).
source_limiter, source_keys = None, {}
if max_queries_per_source > 0:
    source_keys = lock_store.join_sources({source_io.source_key(c) for c in configs})
    source_limiter = parallel.SourceLimiter()

    def refresh_source_shares():
        runs = lock_store.source_runs(source_keys.values())
        for source, share in ss.source_shares(max_queries_per_source, source_keys, runs).items():
            source_limiter.set_limit(source, share)

    try:
        refresh_source_shares()
    except Exception as e:   # the other runs can't be counted: cap this run alone at the whole limit
        print(f"[caps] could not count the runs per source ({str(e)[:200]}); this run takes the whole cap")
        for source in source_keys:
            source_limiter.set_limit(source, max_queries_per_source)

    def _refresh_source_shares_loop():
        while True:
            time.sleep(15)
            try:
                refresh_source_shares()
            except Exception as e:
                print(f"[caps] share refresh failed, keeping the last shares: {str(e)[:200]}")

    threading.Thread(target=_refresh_source_shares_loop, daemon=True, name="source-shares").start()
    print("source caps (this run's share):", {s: source_limiter.limit(s) for s in source_keys})
slots = parallel.WorkSlots(parallelism, set_pool=set_pool, sources=source_limiter)
unit_executor = ThreadPoolExecutor(max_workers=parallelism, thread_name_prefix="unit")

# The per-table worker, retried on transient source errors.
deps = SimpleNamespace(spark=spark, store=store, slots=slots, executor=unit_executor, log=print,
                       params={"table_retries": table_retries, "cursor_lookback_s": cursor_lookback_s,
                               "cursor_skew_s": cursor_skew_s}, guard=assert_lock_held,
                       retry_policy=retry_policy)
worker_fn = table_retry.with_retries(partial(worker.ingest_one, run_id=run_id, deps=deps), retry_policy,
                                     partial(worker._replan, run_id=run_id, deps=deps))

In [ ]:
# Planning (src/lakefed_ingest/planner.py — pure, unit-tested), then the per-table fan-out.
#  0. Sink ownership is checked (invalid client_id; a sink shared by a dedicated row and consolidated
#     rows), then each row's (src_type, load_type) against the support matrix: unsupported pairs and
#     placeholder strategies fail per table (stage 'plan').
#  1. ONE Lakebase query loads every checkpoint. 'full' rows go straight to a reload; 'cursor' rows
#     are routed cursor_seed / schema_reseed / cursor (one high-watermark query per table, in the worker).
#  2. Seeds are capped at max_seeds_per_run; the rest seed in later runs. Every plan then goes
#     through the per-table worker.
import time as _time

t_plan = _time.monotonic()
checkpoints = store.get_checkpoints([c["id"] for c in configs])
plans = planner.route_tables(configs, checkpoints, mixed_sinks=store.mixed_sinks())
unsupported = [p for p in plans if p["route"] == "unsupported"]
plans = [p for p in plans if p["route"] != "unsupported"]
if unsupported:   # e.g. a placeholder (cursor on Oracle or MySQL) or an unknown load_type
    store.record_failed_tables(run_id, [(p["cfg"], p["cp"], p["error"]) for p in unsupported], stage="plan")
    for p in unsupported[:5]:
        c = p["cfg"]
        print(f"[failed@plan] {c['src_database']}.{c['src_schema']}.{c['src_table']}: {p['error']}")
for p in plans:
    if p["route"] == "schema_reseed":
        c = p["cfg"]
        print(f"[schema] {c['src_database']}.{c['src_schema']}.{c['src_table']}: select_list added {p['added']}")

plans, deferred_seeds = planner.apply_seed_budget(plans, max_seeds_per_run)   # the rest seed in later runs
planning = {"seeds_deferred": len(deferred_seeds), "plan_s": round(_time.monotonic() - t_plan, 1),
            "routes": {k: sum(1 for p in plans if p["route"] == k)
                       for k in ("full", "cursor_seed", "cursor", "schema_reseed")},
            "unsupported": len(unsupported)}
print("plan:", json.dumps(planning, default=str))

# Per-table work: every unit draws on `slots`.
results = parallel.run_parallel(plans, worker_fn, parallelism, set_pool=None,
                                item_id=lambda p: f"{p['cfg']['src_database']}.{p['cfg']['src_schema']}.{p['cfg']['src_table']}")
unit_executor.shutdown(wait=True)
summary = parallel.summarize(results)
n_unsupported = len(unsupported)
summary["total"] += n_unsupported
summary["failed"] += n_unsupported
if n_unsupported:
    summary["by_outcome"]["failed_plan"] = n_unsupported
by = summary.get("by_outcome", {})
# Per source: this run's last share of max_queries_per_source and the most queries it ran at once.
source_caps = {s: {"share": source_limiter.limit(s), "peak": source_limiter.peak.get(s, 0)}
               for s in source_keys} if source_limiter is not None else {}
reseeded = by.get("reseed_enqueued", 0)
store.finish_run(
    run_id,
    total=summary["total"], ok=summary["ok"], failed=summary["failed"],
    skipped=by.get("unchanged", 0), reseeded=reseeded,
    status="completed" if summary["failed"] == 0 else "failed",
    detail=json.dumps({"parallelism": parallelism, "peak_units": slots.peak, "planning": planning,
                       "source_caps": source_caps, "table_retries": retry_policy.summary()}, default=str),
)

summary["run_id"] = run_id
summary["task_collection"] = task_collection
summary["parallelism"] = parallelism
summary["peak_units"] = slots.peak
summary["source_caps"] = source_caps
summary["planning"] = planning
summary["table_retries"] = retry_policy.summary()
print(json.dumps(summary, indent=2, default=str))

failures = [r for r in results if r["status"] == "failed"]
if failures:
    print("failures:", json.dumps(failures, default=str)[:2000])

lock_store.close()  # closing the session releases the collection lock (safe even if it already dropped)

dbutils.notebook.exit(json.dumps(summary, default=str))